# Housing Price Regression with PCA and Feature Selection

## Overview

This project explores dimensionality reduction and feature selection techniques in machine learning using the House Prices dataset. Linear regression models were developed and evaluated before and after applying Principal Component Analysis (PCA) and variance threshold feature selection methods.

The project compares how preprocessing and feature engineering techniques influence regression model performance, interpretability, and dimensionality.

---

## Techniques Used

- Linear Regression
- PCA (90% variance retention)
- Variance Threshold Feature Selection
- Dummy Variable Encoding
- Missing Value Imputation
- MinMax Scaling
- Train/Test Split
- R² and RMSE Evaluation

---

## Dataset

The House Prices dataset was used to predict home sale prices based on multiple housing characteristics and engineered numerical features.

---

## Objective

The goal of this project was to evaluate how dimensionality reduction and feature selection impact machine learning model performance while reducing dataset complexity.

## Part 1: PCA and Variance Threshold in Linear Regression

### Load Training Dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("train.csv")

df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## Train/Test Split Before Preprocessing

The dataset is divided into training and testing subsets before fitting preprocessing transformations. This prevents information from the test data from influencing imputation, feature selection, or scaling decisions.


In [2]:

from sklearn.model_selection import train_test_split

# Separate predictors and target
X = df.drop(columns=["Id", "SalePrice"])
y = df["SalePrice"]

# Split before fitting preprocessing transformations
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training features:", X_train_raw.shape)
print("Testing features:", X_test_raw.shape)
print("Training targets:", y_train.shape)
print("Testing targets:", y_test.shape)


Training features: (1168, 79)
Testing features: (292, 79)
Training targets: (1168,)
Testing targets: (292,)


The dataset was successfully loaded and inspected to confirm structure and data types.

In [3]:
# Identify columns with more than 40% missing values
# using the training data only
missing_percent = X_train_raw.isnull().mean()

high_missing = missing_percent[missing_percent > 0.40].index

# Apply the same column selection to both datasets
X_train_filtered = X_train_raw.drop(columns=high_missing).copy()
X_test_filtered = X_test_raw.drop(columns=high_missing).copy()

print("Columns removed:", list(high_missing))
print("Training shape:", X_train_filtered.shape)
print("Testing shape:", X_test_filtered.shape)
print("Remaining missing values in training data:")
print(X_train_filtered.isnull().sum().sort_values(ascending=False).head())

Columns removed: ['Alley', 'MasVnrType', 'FireplaceQu', 'PoolQC', 'Fence', 'MiscFeature']
Training shape: (1168, 73)
Testing shape: (292, 73)
Remaining missing values in training data:
LotFrontage     217
GarageQual       64
GarageType       64
GarageYrBlt      64
GarageFinish     64
dtype: int64


Columns with more than 40% missing values were identified using the training data only. The same columns were then removed from both training and testing datasets to maintain consistent predictor sets.

This prevents information from the held-out test data from influencing feature selection. Remaining missing values will be handled through imputation fitted on the training data.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
import pandas as pd

# Identify feature types from the training data
num_cols = X_train_filtered.select_dtypes(
    include=["number"]
).columns.tolist()

cat_cols = X_train_filtered.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

# Numerical preprocessing: median imputation
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing: mode imputation and encoding
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        drop="first",
        sparse_output=False
    ))
])

# Combine both preprocessing workflows
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, num_cols),
    ("categorical", categorical_pipeline, cat_cols)
])

# Fit on training data only
X_train_processed = preprocessor.fit_transform(X_train_filtered)

# Apply the learned transformations to test data
X_test_processed = preprocessor.transform(X_test_filtered)

# Recover feature names for interpretation
feature_names = preprocessor.get_feature_names_out()

X_train = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train_filtered.index
)

X_test = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test_filtered.index
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Remaining missing values:", X_train.isna().sum().sum())
print("Feature columns match:", X_train.columns.equals(X_test.columns))
print("Testing missing values:", X_test.isna().sum().sum())

Training shape: (1168, 227)
Testing shape: (292, 227)
Remaining missing values: 0
Feature columns match: True
Testing missing values: 0


C:\Users\sfnor\miniconda3\envs\ds_main\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [13, 27] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Missing values and categorical variables were processed using separate scikit-learn pipelines. Numerical features were imputed using the median, while categorical features were imputed using the most frequent category and converted to indicator variables using one-hot encoding.

The preprocessing transformations were fitted exclusively on the training data and then applied to the testing data. This prevents the held-out observations from influencing imputation statistics or the learned categorical encoding.

The resulting feature matrices contain 227 predictors, with no remaining missing values. Categories encountered only in the test data are handled using the encoder's `handle_unknown="ignore"` setting, which preserves consistent feature dimensions.

In [5]:

# Examine categorical expansion learned from training data

encoder = preprocessor.named_transformers_[
    "categorical"
].named_steps["encoder"]

category_counts = pd.DataFrame({
    "Feature": cat_cols,
    "Training Categories": [
        len(categories)
        for categories in encoder.categories_
    ]
})

# Count the encoded columns generated for each feature
encoded_names = encoder.get_feature_names_out(cat_cols)

category_counts["Encoded Columns"] = [
    sum(name.startswith(f"{col}_") for name in encoded_names)
    for col in cat_cols
]

category_counts = category_counts.sort_values(
    "Encoded Columns",
    ascending=False
)

category_counts.head(12)


,Feature,Training Categories,Encoded Columns
7,Neighborhood,25,24
15,Exterior2nd,16,15
14,Exterior1st,15,14
8,Condition1,9,8
35,SaleType,9,8
9,Condition2,8,7
11,HouseStyle,8,7
29,Functional,7,6
13,RoofMatl,7,6
23,BsmtFinType2,6,5


### Dummy Variable Expansion Analysis

One-hot encoding increased the number of predictors by converting categorical features into numerical indicator variables. Neighborhood contributed the most encoded columns (24), followed by Exterior2nd (15) and Exterior1st (14).

The encoder was fitted exclusively on the training data, so the resulting categories reflect the training observations rather than the full dataset. With the first category dropped from each feature, the final processed dataset contains 227 predictors.

This expansion illustrates how categorical encoding can substantially increase dimensionality, which is relevant when evaluating feature selection and PCA.

## Baseline Linear Regression Model

In [6]:

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Train baseline model using all processed predictors
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)

# Predict on held-out test data
baseline_predictions = baseline_model.predict(X_test)

# Evaluate baseline performance
baseline_r2 = r2_score(y_test, baseline_predictions)
baseline_rmse = np.sqrt(
    mean_squared_error(y_test, baseline_predictions)
)

print("Number of predictors:", X_train.shape[1])
print(f"Baseline R²: {baseline_r2:.4f}")
print(f"Baseline RMSE: ${baseline_rmse:,.2f}")

Number of predictors: 227
Baseline R²: 0.8786
Baseline RMSE: $30,509.66


The baseline linear regression model was trained using 227 preprocessed predictors and evaluated on the held-out test set. It achieved an R² of 0.8786 and an RMSE of approximately $30,510.

The R² indicates that the model accounts for approximately 87.9% of the variation in sale prices within this test set. The RMSE measures the square root of the average squared prediction error, expressed in dollars, and gives greater weight to larger errors.

These results establish the updated baseline for evaluating whether dimensionality reduction or feature selection improves predictive performance. Because the preprocessing workflow was revised, these results should not be interpreted as evidence that any single preprocessing change caused the improvement over the original notebook.

## StandardScaler and PCA (90% Variance Retained)

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Standardize predictors using training data only
standard_scaler = StandardScaler()

X_train_standardized = standard_scaler.fit_transform(X_train)
X_test_standardized = standard_scaler.transform(X_test)

# Fit PCA on standardized training data
# Retain enough components to explain 90% of training variance
standard_pca = PCA(n_components=0.90)

X_train_standard_pca = standard_pca.fit_transform(
    X_train_standardized
)

X_test_standard_pca = standard_pca.transform(
    X_test_standardized
)

# Train linear regression using PCA components
standard_pca_model = LinearRegression()

standard_pca_model.fit(
    X_train_standard_pca,
    y_train
)

# Predict on held-out test data
standard_pca_predictions = standard_pca_model.predict(
    X_test_standard_pca
)

# Evaluate performance
standard_pca_r2 = r2_score(
    y_test,
    standard_pca_predictions
)

standard_pca_rmse = np.sqrt(
    mean_squared_error(y_test, standard_pca_predictions)
)

standard_pca_components = standard_pca.n_components_

print("Original predictors:", X_train.shape[1])
print("PCA components retained:", standard_pca_components)
print(f"Baseline R²: {baseline_r2:.4f}")
print(f"Standardized PCA R²: {standard_pca_r2:.4f}")
print(f"Baseline RMSE: ${baseline_rmse:,.2f}")
print(f"Standardized PCA RMSE: ${standard_pca_rmse:,.2f}")

Original predictors: 227
PCA components retained: 127
Baseline R²: 0.8786
Standardized PCA R²: 0.8409
Baseline RMSE: $30,509.66
Standardized PCA RMSE: $34,938.33


The dataset was split into training and test sets using an 80/20 split. 
The `SalePrice` column was used as the target variable, while all remaining columns were used as predictors. 
A baseline linear regression model was fit on the training data and evaluated on the test data using R² and RMSE. 
This baseline model will be compared against the PCA-transformed model and the high-variance feature model in later sections.

## Unscaled PCA (90% Variance Retained)

In [8]:
# Apply PCA directly to the processed predictors
# without additional feature scaling

unscaled_pca = PCA(n_components=0.90)

X_train_unscaled_pca = unscaled_pca.fit_transform(X_train)
X_test_unscaled_pca = unscaled_pca.transform(X_test)

# Train linear regression on the retained components
unscaled_pca_model = LinearRegression()
unscaled_pca_model.fit(X_train_unscaled_pca, y_train)

# Predict on the held-out test set
unscaled_pca_predictions = unscaled_pca_model.predict(
    X_test_unscaled_pca
)

# Evaluate the unscaled PCA model
unscaled_pca_r2 = r2_score(
    y_test,
    unscaled_pca_predictions
)

unscaled_pca_rmse = np.sqrt(
    mean_squared_error(y_test, unscaled_pca_predictions)
)

unscaled_pca_components = unscaled_pca.n_components_

print("Original predictors:", X_train.shape[1])
print("Unscaled PCA components:", unscaled_pca_components)
print(f"Unscaled PCA R²: {unscaled_pca_r2:.4f}")
print(f"Unscaled PCA RMSE: ${unscaled_pca_rmse:,.2f}")

Original predictors: 227
Unscaled PCA components: 1
Unscaled PCA R²: 0.0635
Unscaled PCA RMSE: $84,754.58


PCA was applied directly to the 227 preprocessed predictors without additional feature scaling. Although the transformation was configured to retain 90% of predictor variance, only one principal component was retained.

The resulting regression model achieved an R² of 0.0635 and an RMSE of approximately $84,755, substantially worse than both the baseline and standardized PCA models.

This result demonstrates the sensitivity of PCA to differences in feature variance. Without standardization, predictors measured on larger numerical scales can dominate the principal components, causing PCA to preserve numerical variation that may not be particularly informative for predicting sale prices.

The comparison also demonstrates that retaining 90% of predictor variance does not necessarily preserve 90% of predictive information.

## Min-Max Scaling and Variance Threshold (> 0.1)

In [9]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import VarianceThreshold

# Fit MinMaxScaler on training data only
minmax_scaler = MinMaxScaler()

X_train_minmax = minmax_scaler.fit_transform(X_train)
X_test_minmax = minmax_scaler.transform(X_test)

# Select features exceeding the variance threshold
variance_selector = VarianceThreshold(threshold=0.10)

X_train_selected = variance_selector.fit_transform(
    X_train_minmax
)

X_test_selected = variance_selector.transform(
    X_test_minmax
)

# Train linear regression using selected features
variance_model = LinearRegression()
variance_model.fit(X_train_selected, y_train)

# Predict on held-out test data
variance_predictions = variance_model.predict(
    X_test_selected
)

# Evaluate performance
variance_r2 = r2_score(y_test, variance_predictions)

variance_rmse = np.sqrt(
    mean_squared_error(y_test, variance_predictions)
)

variance_features = X_train_selected.shape[1]

print("Original predictors:", X_train.shape[1])
print("Selected predictors:", variance_features)
print(f"VarianceThreshold R²: {variance_r2:.4f}")
print(f"VarianceThreshold RMSE: ${variance_rmse:,.2f}")

Original predictors: 227
Selected predictors: 40
VarianceThreshold R²: 0.6481
VarianceThreshold RMSE: $51,952.01


MinMaxScaler was fitted on the training predictors to rescale features according to their observed minimum and maximum values. VarianceThreshold was then applied with a threshold of 0.10 to remove features with relatively low variance in the scaled training data.

This reduced the number of predictors from 227 to 40. The resulting linear regression model achieved an R² of 0.6481 and an RMSE of approximately $51,952.

Although this approach reduced dimensionality by approximately 82%, it performed substantially worse than the baseline model. VarianceThreshold evaluates feature variability without considering the relationship between each predictor and SalePrice, so features with low variance may still contain useful predictive information.

The threshold of 0.10 was an exploratory choice rather than a value selected through cross-validation or hyperparameter optimization.

## Model Comparison

In [10]:
# Compare models using the same held-out test set

results = pd.DataFrame({
    "Model": [
        "Baseline Linear Regression",
        "StandardScaler + PCA",
        "Unscaled PCA",
        "MinMaxScaler + VarianceThreshold"
    ],
    "Predictors_or_Components": [
        X_train.shape[1],
        standard_pca_components,
        unscaled_pca_components,
        variance_features
    ],
    "R2": [
        baseline_r2,
        standard_pca_r2,
        unscaled_pca_r2,
        variance_r2
    ],
    "RMSE": [
        baseline_rmse,
        standard_pca_rmse,
        unscaled_pca_rmse,
        variance_rmse
    ]
})

results.sort_values("R2", ascending=False).round(4)

,Model,Predictors_or_Components,R2,RMSE
0,Baseline Linear Regression,227,0.8786,30509.6555
1,StandardScaler + PCA,127,0.8409,34938.3274
3,MinMaxScaler + VarianceThreshold,40,0.6481,51952.0147
2,Unscaled PCA,1,0.0635,84754.5801


### Interpretation of Model Comparison

The baseline linear regression model achieved the strongest predictive performance, with an R² of 0.8786 and an RMSE of approximately $30,510. This indicates that retaining all 227 preprocessed predictors produced better results on the held-out test set than either dimensionality reduction approach.

StandardScaler followed by PCA retained 127 components and achieved an R² of 0.8409. Although this reduced dimensionality by approximately 44%, it also resulted in a moderate decline in predictive performance.

The MinMaxScaler and VarianceThreshold approach retained only 40 predictors but produced a lower R² of 0.6481. The unscaled PCA model performed worst, retaining only one component and achieving an R² of 0.0635.

These findings demonstrate that reducing dimensionality does not necessarily improve predictive accuracy. PCA preserves directions of predictor variance without directly considering the target variable, while VarianceThreshold removes features based on variability rather than predictive relevance.

The results also illustrate the importance of feature scaling when using PCA. Standardizing the predictors before PCA produced a substantially different representation of the data and considerably better predictive performance than applying PCA without additional scaling.

Overall, the baseline was the strongest model for this particular train/test split. Further evaluation using cross-validation would be needed to determine how consistently these findings generalize to other data partitions.

## Summary of Findings and Limitations

This project compared baseline linear regression with three dimensionality-reduction and feature-selection approaches using the Ames housing dataset. All models were evaluated using the same 80/20 train/test split, with preprocessing transformations fitted on the training data.

The baseline linear regression model achieved the strongest predictive performance, with an R² of 0.8786 and an RMSE of approximately $30,510. StandardScaler followed by PCA retained 127 of the original 227 predictors as principal components, achieving an R² of 0.8409. Although this reduced dimensionality by approximately 44%, it also reduced predictive performance.

The unscaled PCA experiment retained only one principal component despite preserving 90% of predictor variance. Its substantially weaker performance demonstrated how differences in feature scale can influence PCA. The MinMaxScaler and VarianceThreshold approach reduced the dataset to 40 predictors but also performed worse than the baseline.

The most important finding was that dimensionality reduction did not necessarily improve predictive performance. PCA identifies directions of predictor variance without considering the target variable, while VarianceThreshold removes features according to variability rather than predictive relevance.

### Limitations

The results were obtained using a single train/test split and were not validated through cross-validation. The 90% PCA variance retention setting, 40% missing-value threshold, and 0.10 variance threshold were exploratory choices rather than optimized hyperparameters.

The initial preprocessing steps were fitted on training data only, reducing the risk of data leakage. However, the comparison does not establish that the baseline would consistently outperform the other methods across different samples or datasets.

The analysis also used ordinary linear regression without regularization, and the models were not evaluated for coefficient stability, residual patterns, or sensitivity to outliers.

### Reflection

Revisiting this project helped me better understand the distinction between implementing a preprocessing technique and understanding when it is appropriate. Initially, I was focused on applying scikit-learn methods and comparing model outputs. Returning to the notebook with a stronger understanding of scaling and PCA allowed me to identify methodological weaknesses, correct the preprocessing workflow, and reconsider the interpretation of the results.

The experience reinforced the importance of understanding what a transformation actually does before deciding whether it is appropriate for a particular modeling problem.